# TRICARE 파인튜닝 데이터셋 생성
## 목표: 총 10,000개 JSONL 데이터셋

| 레이어 | 방법 | 목표 수량 |
|--------|------|----------|
| Layer 1 | Hugging Face 공개 데이터셋 필터링 | 7,000~8,000개 |
| Layer 2 | TRICARE PDF/CSV 청크 기반 GPT-4 자동생성 | 1,500~2,000개 |
| Layer 3 | 패턴 데이터 직접 작성 (면책·거절·차단) | 200~300개 |

**실행 순서**: 셀을 위에서 아래로 순서대로 실행


## 0. 패키지 설치 및 환경 설정

In [ ]:
!pip install datasets openai pandas tqdm -q


In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
from pathlib import Path

# OpenAI API 키 설정
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # ← 본인 키로 교체

OUTPUT_DIR = Path("./finetune_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ 환경 설정 완료")
print(f"출력 폴더: {OUTPUT_DIR.resolve()}")


---
## Layer 1 — Hugging Face 공개 데이터셋 필터링
보험/의료비용/coverage 관련 Q&A만 추출해서 TRICARE SFT 형식으로 변환

### 사용 데이터셋
| 데이터셋 | 전체 수량 | 필터링 후 예상 |
|---------|----------|--------------|
| `gbharti/finance-alpaca` | 68k | ~3,000개 |
| `lavita/ChatDoctor-HealthCareMagic-100k` | 100k | ~4,000개 |
| `keivalya/MedQuad-MedicalQnADataset` | 47k | ~1,000개 |


In [ ]:
from datasets import load_dataset

# ── 1-A. finance-alpaca 로드 및 필터링 ─────────────────

print("finance-alpaca 로드 중... (약 1~2분 소요)")
ds_finance = load_dataset("gbharti/finance-alpaca", split="train")
df_finance = pd.DataFrame(ds_finance)
print(f"전체 행 수: {len(df_finance)}")
print(df_finance.columns.tolist())
df_finance.head(3)


In [ ]:
# 보험/의료비용 관련 키워드 필터
INSURANCE_KEYWORDS = [
    "insurance", "coverage", "deductible", "copay", "co-pay",
    "premium", "benefit", "claim", "reimbursement", "out-of-pocket",
    "health plan", "medical cost", "prior authorization",
    "tricare", "medicare", "medicaid", "health insurance"
]

def has_insurance_keyword(text: str) -> bool:
    text_lower = str(text).lower()
    return any(kw in text_lower for kw in INSURANCE_KEYWORDS)

# instruction 또는 output에 키워드가 있는 행만 추출
mask = df_finance["instruction"].apply(has_insurance_keyword) |        df_finance["output"].apply(has_insurance_keyword)

df_finance_filtered = df_finance[mask].reset_index(drop=True)
print(f"필터링 후: {len(df_finance_filtered)}행")
df_finance_filtered.head(3)


In [ ]:
# SFT JSONL 형식으로 변환
# 형식: {"messages": [{"role":"system",...}, {"role":"user",...}, {"role":"assistant",...}]}

SYSTEM_PROMPT = (
    "You are a TRICARE health benefits specialist. "
    "This system is designed for OCONUS beneficiaries, primarily Korean residents and USFK personnel. "
    "Answer ONLY based on provided documents. "
    "Never recommend specific plans. "
    "Always include a disclaimer that actual coverage depends on individual contracts."
)

def finance_row_to_jsonl(row) -> dict:
    user_content = row["instruction"]
    if row.get("input", "").strip():
        user_content += f"

{row['input']}"
    return {
        "messages": [
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": user_content},
            {"role": "assistant","content": row["output"]},
        ]
    }

layer1a_data = [finance_row_to_jsonl(row) for _, row in df_finance_filtered.iterrows()]

# 저장
path_1a = OUTPUT_DIR / "layer1a_finance.jsonl"
with open(path_1a, "w", encoding="utf-8") as f:
    for item in layer1a_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Layer1-A 저장 완료: {len(layer1a_data)}개 → {path_1a}")


In [ ]:
# ── 1-B. ChatDoctor-HealthCareMagic 로드 및 필터링 ────────

print("ChatDoctor-HealthCareMagic-100k 로드 중... (약 2~3분 소요)")
ds_chat = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
df_chat = pd.DataFrame(ds_chat)
print(f"전체 행 수: {len(df_chat)}")
print(df_chat.columns.tolist())
df_chat.head(2)


In [ ]:
# 의료비용/보험 관련 키워드 필터
MEDICAL_COST_KEYWORDS = [
    "insurance", "coverage", "cost", "payment", "bill",
    "deductible", "copay", "claim", "reimburse", "benefit",
    "covered", "not covered", "out of pocket", "premium"
]

def has_cost_keyword(text: str) -> bool:
    text_lower = str(text).lower()
    return any(kw in text_lower for kw in MEDICAL_COST_KEYWORDS)

# input(사용자 질문) 또는 output(의사 답변)에 비용 키워드 있는 행
mask_chat = df_chat["input"].apply(has_cost_keyword) |             df_chat["output"].apply(has_cost_keyword)

df_chat_filtered = df_chat[mask_chat].reset_index(drop=True)

# 너무 많으면 4,000개로 제한 (품질 유지)
if len(df_chat_filtered) > 4000:
    df_chat_filtered = df_chat_filtered.sample(4000, random_state=42).reset_index(drop=True)

print(f"필터링 후: {len(df_chat_filtered)}행")


In [ ]:
def chatdoctor_row_to_jsonl(row) -> dict:
    return {
        "messages": [
            {"role": "system",   "content": SYSTEM_PROMPT},
            {"role": "user",     "content": str(row["input"])},
            {"role": "assistant","content": str(row["output"])},
        ]
    }

layer1b_data = [chatdoctor_row_to_jsonl(row) for _, row in df_chat_filtered.iterrows()]

path_1b = OUTPUT_DIR / "layer1b_chatdoctor.jsonl"
with open(path_1b, "w", encoding="utf-8") as f:
    for item in layer1b_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Layer1-B 저장 완료: {len(layer1b_data)}개 → {path_1b}")


In [ ]:
# ── 1-C. MedQuad 로드 및 필터링 ──────────────────────────

print("MedQuad-MedicalQnADataset 로드 중...")
ds_med = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
df_med = pd.DataFrame(ds_med)
print(f"전체 행 수: {len(df_med)}")
print(df_med.columns.tolist())
df_med.head(2)


In [ ]:
# 보험/비용 관련 FAQ만 추출
mask_med = df_med["Question"].apply(has_insurance_keyword) |            df_med["Answer"].apply(has_insurance_keyword)

df_med_filtered = df_med[mask_med].reset_index(drop=True)

if len(df_med_filtered) > 1000:
    df_med_filtered = df_med_filtered.sample(1000, random_state=42).reset_index(drop=True)

print(f"필터링 후: {len(df_med_filtered)}행")


In [ ]:
def medquad_row_to_jsonl(row) -> dict:
    return {
        "messages": [
            {"role": "system",   "content": SYSTEM_PROMPT},
            {"role": "user",     "content": str(row["Question"])},
            {"role": "assistant","content": str(row["Answer"])},
        ]
    }

layer1c_data = [medquad_row_to_jsonl(row) for _, row in df_med_filtered.iterrows()]

path_1c = OUTPUT_DIR / "layer1c_medquad.jsonl"
with open(path_1c, "w", encoding="utf-8") as f:
    for item in layer1c_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Layer1-C 저장 완료: {len(layer1c_data)}개 → {path_1c}")

# Layer 1 전체 합산
total_l1 = len(layer1a_data) + len(layer1b_data) + len(layer1c_data)
print(f"\n📊 Layer 1 합계: {total_l1}개")


---
## Layer 2 — TRICARE PDF/CSV 청크 기반 GPT-4 자동생성
기존 ChromaDB에서 청크를 불러와서 청크당 Q&A 5개씩 GPT-4로 생성


In [ ]:
# ChromaDB에서 전체 청크 로드
# tricare_core.py의 벡터 DB 경로와 동일하게 맞출 것
import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PERSIST_TEXT    = './chroma_db'   # tricare_core.py와 동일 경로
COLLECTION_TEXT = 'tricare_rag'

print("ChromaDB 로드 중...")
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)

vector_store = Chroma(
    collection_name=COLLECTION_TEXT,
    embedding_function=embedding_model,
    persist_directory=PERSIST_TEXT
)

raw = vector_store._collection.get(include=['documents', 'metadatas'])
chunks = list(zip(raw['documents'], raw['metadatas']))
print(f"✅ 청크 {len(chunks)}개 로드 완료")


In [ ]:
from openai import OpenAI
import time

client = OpenAI()  # OPENAI_API_KEY 환경변수에서 자동 로드

def generate_qa_from_chunk(chunk_text: str, source: str, page: str) -> list[dict]:
    """
    청크 하나로부터 Q&A 5개 생성.
    반환: JSONL 형식 dict 리스트
    """
    prompt = f"""아래 TRICARE 공식 문서 청크를 읽고,
이 내용에 대해 한국 거주 외국인(USFK 포함)이 실제로 물어볼 법한
질문 5개와 모범 답변을 만들어줘.

규칙:
1. 답변은 반드시 출처 [{source}, p.{page}] 형식으로 시작
2. 답변 끝에 면책 고지 포함: "⚠️ 실제 적용은 개인 계약 조건에 따라 다를 수 있습니다. 보험사에 직접 확인하세요."
3. 보험 추천/권유 표현 절대 금지
4. 보험 용어는 한국어(영어) 병기: 본인부담금(Copay), 공제액(Deductible) 등
5. 질문은 한국어 또는 영어로 섞어서

[문서 청크]
{chunk_text[:800]}

아래 JSON 배열 형식으로만 응답 (다른 텍스트 없이):
[
  {{"question": "...", "answer": "..."}},
  ...
]
"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=2000,
        )
        raw_text = resp.choices[0].message.content.strip()
        # ```json 펜스 제거
        raw_text = raw_text.replace("```json", "").replace("```", "").strip()
        qa_list = json.loads(raw_text)

        # JSONL 형식으로 변환
        results = []
        for qa in qa_list:
            results.append({
                "messages": [
                    {"role": "system",   "content": SYSTEM_PROMPT},
                    {"role": "user",     "content": qa["question"]},
                    {"role": "assistant","content": qa["answer"]},
                ]
            })
        return results

    except Exception as e:
        print(f"  ⚠️ 생성 실패: {e}")
        return []

print("✅ Q&A 생성 함수 준비 완료")


In [ ]:
# 전체 청크 대상으로 Q&A 생성
# 예상 소요: 청크 400개 × 약 3초 = 약 20분
# API 비용 절감을 위해 청크 300개까지만 처리 (필요 시 늘릴 것)

MAX_CHUNKS = 400  # 필요 시 조정
layer2_data = []
failed = 0

print(f"총 {min(MAX_CHUNKS, len(chunks))}개 청크 처리 시작...")

for i, (doc_text, meta) in enumerate(tqdm(chunks[:MAX_CHUNKS])):
    source = meta.get('source_file', meta.get('source', 'TRICARE Document'))
    page   = meta.get('page', '')

    qa_items = generate_qa_from_chunk(doc_text, source, page)
    layer2_data.extend(qa_items)

    # API 부하 방지
    time.sleep(0.5)

    # 중간 저장 (100개마다)
    if (i + 1) % 100 == 0:
        path_tmp = OUTPUT_DIR / f"layer2_checkpoint_{i+1}.jsonl"
        with open(path_tmp, "w", encoding="utf-8") as f:
            for item in layer2_data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"  💾 체크포인트 저장: {len(layer2_data)}개")

# 최종 저장
path_2 = OUTPUT_DIR / "layer2_chunk_generated.jsonl"
with open(path_2, "w", encoding="utf-8") as f:
    for item in layer2_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n✅ Layer 2 저장 완료: {len(layer2_data)}개 → {path_2}")


---
## Layer 3 — 패턴 데이터 직접 작성
면책고지 / 추천거절 / 개인정보차단 패턴은 품질이 가장 중요해서 직접 작성.
아래 시드 데이터 기반으로 GPT-4가 유사 패턴 추가 생성.


In [ ]:
# 시드 데이터 — 직접 작성한 핵심 패턴
# 이 데이터는 파인튜닝의 "행동 고정" 역할을 함
# 직접 추가/수정 가능

SEED_DATA = [

    # ── 패턴 1: Coverage 답변 + 면책고지 ──────────────────
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "TRICARE Prime에서 한국에서 정신건강 상담을 받을 수 있나요?"},
            {"role": "assistant", "content": (
                "[Overseas_HB(해외 프로그램 안내서).pdf, p.10 기준]\n\n"
                "한국(OCONUS)에서 TRICARE Prime 수혜자는 TRICARE Overseas Program(TOP)을 통해 "
                "정신건강 상담(Mental Health) 서비스를 받을 수 있습니다.\n\n"
                "• TRICARE는 한국에서 1차 보험자(Primary Payer)로 적용됩니다.\n"
                "• 해외에서는 먼저 비용을 지불한 후 청구서를 제출하는 방식(Pay-up-front)입니다.\n"
                "• 청구 기한: 서비스 제공일로부터 3년 이내\n\n"
                "⚠️ 실제 적용은 개인 계약 조건에 따라 다를 수 있습니다. "
                "tricare-overseas.com에서 직접 확인하세요."
            )},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "What is the copay for outpatient mental health visits under TRICARE Select?"},
            {"role": "assistant", "content": (
                "[Health_Plan_Costs.csv / TRICARE Select 기준]\n\n"
                "TRICARE Select outpatient mental health visit copay:\n\n"
                "• Group A (Active Duty Family Members): $0 copay\n"
                "• Group B (Retirees and their families): 본인부담금(Copay) $25 per visit\n"
                "• 공제액(Deductible) 충족 후 위 copay 적용됩니다.\n\n"
                "⚠️ This information is based on standard plan costs. "
                "Actual costs may vary. Please verify with TRICARE directly."
            )},
        ]
    },

    # ── 패턴 2: 보험 추천 요청 거절 ───────────────────────
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "TRICARE Prime이랑 Select 중에 어떤 게 더 좋아요?"},
            {"role": "assistant", "content": (
                "본 서비스는 특정 보험 플랜을 추천하거나 가입을 권유하지 않습니다.\n\n"
                "다만, 두 플랜의 주요 차이를 객관적으로 안내해 드릴 수 있습니다:\n\n"
                "• TRICARE Prime: 주치의(PCM) 지정 필요, 본인부담금(Copay) 낮음, 의뢰(Referral) 필요\n"
                "• TRICARE Select: 네트워크 내(In-network) 의사 자유 선택, "
                "공제액(Deductible) 있음, 사전승인(Prior Authorization) 불필요한 경우 많음\n\n"
                "어느 플랜이 적합한지는 개인 상황에 따라 다르므로, "
                "TRICARE 공식 사이트(tricare.mil) 또는 담당 설계사에게 직접 문의하세요."
            )},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Which TRICARE plan do you recommend for someone living in Korea?"},
            {"role": "assistant", "content": (
                "This service does not recommend or endorse specific insurance plans.\n\n"
                "For beneficiaries living in Korea (OCONUS), the applicable program is "
                "TRICARE Overseas Program (TOP). Key options include:\n\n"
                "• TRICARE Prime Overseas: Requires enrollment, lower out-of-pocket costs\n"
                "• TRICARE Select Overseas: No enrollment required, more provider flexibility\n\n"
                "The best choice depends on your beneficiary category (Active Duty, Retired, etc.) "
                "and personal circumstances. Please consult tricare-overseas.com for details."
            )},
        ]
    },

    # ── 패턴 3: 개인 건강정보 입력 차단 ──────────────────
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "저 우울증 진단받았는데 TRICARE에서 치료 커버돼요?"},
            {"role": "assistant", "content": (
                "개인 진단명이나 건강 정보는 이 서비스에서 필요하지 않으며 저장되지 않습니다.\n\n"
                "플랜 레벨 기준으로 안내해 드리겠습니다:\n\n"
                "TRICARE는 일반적으로 정신건강(Mental Health) 서비스를 보장하며, "
                "여기에는 외래 심리치료(Outpatient Psychotherapy), "
                "입원 정신건강 치료(Inpatient Mental Health) 등이 포함됩니다.\n\n"
                "구체적인 보장 범위(Coverage)는 가입하신 플랜과 지역에 따라 다릅니다. "
                "플랜명(예: TRICARE Prime, Select)을 알려주시면 더 자세히 안내해 드릴 수 있습니다.\n\n"
                "⚠️ 실제 보장 여부는 tricare.mil 또는 담당 케이스 매니저에게 확인하세요."
            )},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "My insurance ID is 123-456-789 and I have PTSD. Am I covered?"},
            {"role": "assistant", "content": (
                "Personal insurance IDs and diagnosis information are not required by this service "
                "and are not stored or processed here.\n\n"
                "Regarding mental health coverage under TRICARE:\n\n"
                "TRICARE covers a range of mental health services including outpatient psychotherapy, "
                "inpatient mental health treatment, and medication management. "
                "PTSD treatment is generally covered under TRICARE's mental health benefits.\n\n"
                "For your specific plan details, please provide your plan type "
                "(e.g., TRICARE Prime, Select) and I can give more specific information.\n\n"
                "⚠️ Please verify actual coverage with TRICARE directly at tricare.mil."
            )},
        ]
    },

    # ── 패턴 4: OCONUS 특화 질문 ─────────────────────────
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "한국에서 병원 가면 Medicare가 적용되나요?"},
            {"role": "assistant", "content": (
                "[Overseas_HB(해외 프로그램 안내서).pdf 기준]\n\n"
                "아니요. Medicare는 해외(OCONUS) 의료비를 보장하지 않습니다.\n\n"
                "한국 거주 수혜자의 경우:\n"
                "• TRICARE가 1차 보험자(Primary Payer)로 적용됩니다.\n"
                "• Medicare는 해외에서 적용되지 않습니다.\n"
                "• 다만 Medicare Part B에 적극적으로 등록되어 있어야 "
                "TRICARE For Life(TFL) 혜택 유지가 가능합니다.\n\n"
                "⚠️ 실제 적용은 개인 수혜자 카테고리(현역/퇴역 등)에 따라 다릅니다. "
                "tricare-overseas.com에서 확인하세요."
            )},
        ]
    },

    # ── 패턴 5: 청구 절차 안내 ───────────────────────────
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "한국에서 치료받은 후 TRICARE에 어떻게 청구하나요?"},
            {"role": "assistant", "content": (
                "[TOP_Handbook_AUG_2023_FINAL_092223_508_1.pdf 기준]\n\n"
                "해외(OCONUS) TRICARE 청구 절차:\n\n"
                "1. 먼저 의료비 전액을 본인이 지불합니다 (Pay-up-front 방식)\n"
                "2. 영수증, 진단서, 치료 기록 등 서류를 준비합니다\n"
                "3. TRICARE Overseas Program 청구 양식을 작성합니다\n"
                "4. tricare-overseas.com을 통해 온라인 제출 또는 우편 제출\n"
                "5. 청구 기한: 서비스 제공일로부터 3년(36개월) 이내\n\n"
                "⚠️ 청구 서류 요건은 변경될 수 있습니다. "
                "최신 정보는 tricare-overseas.com에서 확인하세요."
            )},
        ]
    },
]

print(f"✅ 시드 데이터 {len(SEED_DATA)}개 준비 완료")


In [ ]:
# GPT-4로 시드 데이터 기반 유사 패턴 증강
# 시드 7개 × 약 40개 생성 = 약 280개

def augment_from_seed(seed_item: dict, n: int = 40) -> list[dict]:
    """시드 Q&A 1개로부터 유사한 패턴 n개 생성"""
    example_q = seed_item["messages"][1]["content"]
    example_a = seed_item["messages"][2]["content"]

    prompt = f"""아래 TRICARE 보험 Q&A 예시를 참고해서,
동일한 응답 규칙을 따르는 유사한 질문-답변 쌍 {n}개를 만들어줘.

응답 규칙:
- 출처 [파일명, p.페이지] 형식 포함
- 면책 고지 포함 (⚠️로 시작)
- 보험 추천/권유 표현 절대 금지
- 개인 진단명/보험번호 요구 금지
- 보험 용어 한국어(영어) 병기
- 질문은 한국어/영어 섞어서 작성

[예시 질문]
{example_q}

[예시 답변]
{example_a}

아래 JSON 배열 형식으로만 응답:
[
  {{"question": "...", "answer": "..."}},
  ...
]
"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8,
            max_tokens=4000,
        )
        raw_text = resp.choices[0].message.content.strip()
        raw_text = raw_text.replace("```json", "").replace("```", "").strip()
        qa_list = json.loads(raw_text)

        return [
            {
                "messages": [
                    {"role": "system",   "content": SYSTEM_PROMPT},
                    {"role": "user",     "content": qa["question"]},
                    {"role": "assistant","content": qa["answer"]},
                ]
            }
            for qa in qa_list
        ]
    except Exception as e:
        print(f"  ⚠️ 증강 실패: {e}")
        return []


layer3_data = list(SEED_DATA)  # 시드 먼저 포함

print("시드 기반 증강 시작...")
for i, seed in enumerate(tqdm(SEED_DATA)):
    augmented = augment_from_seed(seed, n=40)
    layer3_data.extend(augmented)
    print(f"  [{i+1}/{len(SEED_DATA)}] +{len(augmented)}개 생성 (누계: {len(layer3_data)}개)")
    time.sleep(1)

path_3 = OUTPUT_DIR / "layer3_patterns.jsonl"
with open(path_3, "w", encoding="utf-8") as f:
    for item in layer3_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n✅ Layer 3 저장 완료: {len(layer3_data)}개 → {path_3}")


---
## 최종 병합 — 전체 데이터셋 합치기


In [ ]:
import random

all_data = []

for jsonl_file in [
    OUTPUT_DIR / "layer1a_finance.jsonl",
    OUTPUT_DIR / "layer1b_chatdoctor.jsonl",
    OUTPUT_DIR / "layer1c_medquad.jsonl",
    OUTPUT_DIR / "layer2_chunk_generated.jsonl",
    OUTPUT_DIR / "layer3_patterns.jsonl",
]:
    if jsonl_file.exists():
        with open(jsonl_file, "r", encoding="utf-8") as f:
            items = [json.loads(line) for line in f if line.strip()]
        all_data.extend(items)
        print(f"  {jsonl_file.name}: {len(items)}개")
    else:
        print(f"  ⚠️ 파일 없음: {jsonl_file.name}")

print(f"\n병합 전 합계: {len(all_data)}개")

# 중복 제거 (user 질문 기준)
seen = set()
deduped = []
for item in all_data:
    key = item["messages"][1]["content"][:100]
    if key not in seen:
        seen.add(key)
        deduped.append(item)

print(f"중복 제거 후: {len(deduped)}개")

# 셔플
random.seed(42)
random.shuffle(deduped)

# 최종 저장
final_path = OUTPUT_DIR / "tricare_finetune_final.jsonl"
with open(final_path, "w", encoding="utf-8") as f:
    for item in deduped:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n🎉 최종 데이터셋 저장 완료!")
print(f"   파일: {final_path}")
print(f"   총 데이터 수: {len(deduped)}개")


In [ ]:
# 샘플 확인
print("=== 랜덤 샘플 3개 확인 ===\n")
for i, item in enumerate(random.sample(deduped, 3)):
    print(f"[샘플 {i+1}]")
    print(f"Q: {item['messages'][1]['content'][:100]}...")
    print(f"A: {item['messages'][2]['content'][:200]}...")
    print()
